# Notebook 15 - Complete Feature Engineering Workflow

This notebook takes a raw, messy customer transactions dataset all the way through to a final, ML-ready feature set: Clean Dataset to Engineered Dataset to Selected Features to Final ML-Ready Feature Set.

Run the cells in order from top to bottom, since later steps use columns created in earlier steps.

In [5]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv('customer_transactions_raw.csv')
df.shape

(1000, 12)

**Code Explanation:** The raw dataset is loaded as-is, with no cleaning yet, so its real messiness is visible before any fixes are applied.

## 1. Load Clean Dataset

**Definition:** Loading the raw data and fixing structural problems like wrong data types, inconsistent text, currency symbols, and invalid values before any feature engineering begins.

**Example:** purchase_amount contains values like $41.88 as text instead of a number, gender has entries like Female, female, F, and M mixed together, and age has impossible values like negative numbers.

**Why it is used?** Feature engineering on top of dirty data produces dirty, unreliable features, so cleaning always comes first.

In [6]:
df = df.drop(columns=['notes'])
df['gender'] = df['gender'].str.strip()
df['gender'] = df['gender'].str.lower()
df['gender'] = df['gender'].replace({'female': 'Female', 'f': 'Female', 'male': 'Male', 'm': 'Male'})
df['city'] = df['city'].str.strip()
df['city'] = df['city'].str.title()
df['purchase_amount'] = df['purchase_amount'].astype(str)
df['purchase_amount'] = df['purchase_amount'].str.replace('$', '', regex=False)
df['purchase_amount'] = df['purchase_amount'].astype(float)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce', format='mixed')
df.loc[df['age'] < 10, 'age'] = np.nan
df.loc[df['age'] > 100, 'age'] = np.nan
df.loc[df['annual_income'] < 0, 'annual_income'] = np.nan
df.loc[df['annual_income'] > 500000, 'annual_income'] = np.nan
df.loc[df['purchase_amount'] < 0, 'purchase_amount'] = np.nan
df.loc[df['purchase_amount'] > 5000, 'purchase_amount'] = np.nan
df.loc[df['quantity'] <= 0, 'quantity'] = np.nan
df.loc[df['rating'] < 1, 'rating'] = np.nan
df.loc[df['rating'] > 5, 'rating'] = np.nan
df['age'] = df['age'].fillna(df['age'].median())
df['annual_income'] = df['annual_income'].fillna(df['annual_income'].median())
df['purchase_amount'] = df['purchase_amount'].fillna(df['purchase_amount'].median())
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['rating'] = df['rating'].fillna(df['rating'].median())
df['gender'] = df['gender'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')
df['membership_type'] = df['membership_type'].fillna('Unknown')
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['signup_date'] = df['signup_date'].fillna(df['signup_date'].median())
df.isna().sum()

customer_id        0
age                0
gender             0
annual_income      0
city               0
membership_type    0
purchase_amount    0
quantity           0
signup_date        0
payment_method     0
rating             0
dtype: int64

**Code Explanation:** The notes column was dropped since it was completely empty. gender and city text is standardized, purchase_amount is stripped of dollar signs and converted to a real number, age is converted to numeric, and impossible values like negative income, negative quantity, and out-of-range ratings are treated as missing and filled with the median or a placeholder label.

## 2. Understand Existing Features

**Definition:** Reviewing every existing column, its data type, range, and distribution before deciding what new features to build.

**Example:** Checking that annual_income now ranges from a few thousand to about 126,000 after cleaning, with no more impossible negative values.

**Why it is used?** You cannot engineer good features without first understanding what the raw columns actually represent and how clean they now are.

In [3]:
df.describe()

,customer_id,age,annual_income,purchase_amount,quantity,signup_date,rating
count,1000.000000,1000.000000,1000.00000,1000.00000,1000.000000,1000,1000.000000
mean,100478.341000,37.976000,65855.50817,124.50118,5.026000,2021-07-10 14:24:00,2.971000
min,100001.000000,18.000000,3724.40000,2.73000,1.000000,2019-01-03 00:00:00,1.000000
25%,100240.750000,30.000000,53600.67750,63.02500,3.000000,2020-05-16 00:00:00,2.000000
50%,100479.500000,38.000000,65884.34000,103.90000,5.000000,2021-07-17 12:00:00,3.000000
75%,100717.250000,44.250000,77781.07250,169.00250,7.000000,2022-09-13 06:00:00,4.000000
max,100950.000000,76.000000,126190.62000,511.49000,9.000000,2023-12-30 00:00:00,5.000000
std,274.843253,10.771883,19481.54799,84.09071,2.546332,NaN,1.409307


**Code Explanation:** describe summarizes all numeric columns after cleaning, confirming the ranges now look realistic, with no more negative ages, incomes, or purchase amounts, and 1000 non missing values across the board.

## 3. Identify Potential Feature Gaps

**Definition:** Spotting useful information that is implied by the raw columns but not yet captured as its own feature.

**Example:** The dataset has customer_id repeated across rows, meaning some customers made more than one purchase, but there is no existing feature for how many times a customer has bought, or how long they have been a customer.

**Why it is used?** Identifying gaps early guides exactly which new features are worth building instead of engineering features at random.

In [4]:
df['customer_id'].duplicated().sum()

np.int64(50)

**Code Explanation:** 50 duplicate customer IDs confirm that some customers appear more than once, which is the gap that motivates building customer-level features like total spend and transaction count later on.

## 4. Create Numerical Features

**Definition:** Building new numeric columns from existing numeric columns.

**Example:** Price_Per_Item, the cost of a single item within a purchase.

**Why it is used?** Ratios and derived numbers often reveal patterns that raw totals alone do not show.

In [5]:
df['Price_Per_Item'] = df['purchase_amount'] / df['quantity']
df['Income_To_Purchase_Ratio'] = df['annual_income'] / df['purchase_amount']
df[['purchase_amount', 'quantity', 'Price_Per_Item', 'annual_income', 'Income_To_Purchase_Ratio']].head()

,purchase_amount,quantity,Price_Per_Item,annual_income,Income_To_Purchase_Ratio
0,99.13,1.0,99.130000,80242.98,809.472208
1,113.23,1.0,113.230000,65884.34,581.862934
2,248.19,2.0,124.095000,56285.40,226.783513
3,51.27,9.0,5.696667,81878.74,1597.010728
4,97.97,1.0,97.970000,100566.42,1026.502195


**Code Explanation:** Price_Per_Item divides purchase_amount by quantity to get a per-item cost, and Income_To_Purchase_Ratio compares a customer's income against what they spent in that transaction.

## 5. Create Categorical Features

**Definition:** Grouping numeric columns into readable categories.

**Example:** Income_Bracket groups annual_income into Low, Medium, High, and Very High.

**Why it is used?** Turns continuous ranges into business-friendly groups that are easier to analyze and can help certain models.

In [7]:
income_bins = [0, 40000, 70000, 100000, np.inf]
income_labels = ['Low', 'Medium', 'High', 'Very High']
df['Income_Bracket'] = pd.cut(df['annual_income'], bins=income_bins, labels=income_labels)
age_bins = [0, 25, 40, 60, 100]
age_labels = ['Young Adult', 'Adult', 'Middle Aged', 'Senior']
df['Age_Group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)
df[['annual_income', 'Income_Bracket', 'age', 'Age_Group']].head()

,annual_income,Income_Bracket,age,Age_Group
0,80242.98,High,43.0,Middle Aged
1,65884.34,Medium,41.0,Middle Aged
2,56285.40,Medium,61.0,Senior
3,81878.74,High,38.0,Adult
4,100566.42,Very High,24.0,Young Adult


**Code Explanation:** pd.cut converts annual_income and age into labeled brackets using fixed real-world cutoffs, making both columns easier to read and group by.

## 6. Create Date/Time Features

**Definition:** Extracting useful parts, like year, month, or elapsed time, out of a date column.

**Example:** Tenure_Days, how many days ago a customer signed up relative to the most recent signup date in the data.

**Why it is used?** Raw dates are not directly useful to a model, but the patterns hidden inside them, like how long ago something happened, usually are.

In [8]:
snapshot_date = df['signup_date'].max() + pd.Timedelta(days=1)
df['Signup_Year'] = df['signup_date'].dt.year
df['Signup_Month'] = df['signup_date'].dt.month
df['Tenure_Days'] = (snapshot_date - df['signup_date']).dt.days
df[['signup_date', 'Signup_Year', 'Signup_Month', 'Tenure_Days']].head()

,signup_date,Signup_Year,Signup_Month,Tenure_Days
0,2021-08-22,2021,8,861
1,2020-06-16,2020,6,1293
2,2020-09-28,2020,9,1189
3,2023-09-10,2023,9,112
4,2021-07-01,2021,7,913


**Code Explanation:** Signup_Year and Signup_Month pull the year and month directly out of signup_date, while Tenure_Days measures how many days have passed since each customer signed up, relative to a fixed snapshot point.

## 7. Create Aggregation Features

**Definition:** Summarizing values across groups, like per customer or per membership type, into new columns.

**Example:** Customer_Total_Purchase, the sum of everything a specific customer has spent across all their transactions.

**Why it is used?** Captures customer-level or group-level behavior that a single transaction row cannot show on its own.

In [9]:
membership_avg = df.groupby('membership_type')['purchase_amount'].mean()
df['Avg_Purchase_By_Membership'] = df['membership_type'].map(membership_avg)
customer_total = df.groupby('customer_id')['purchase_amount'].sum()
df['Customer_Total_Purchase'] = df['customer_id'].map(customer_total)
customer_count = df.groupby('customer_id')['customer_id'].count()
df['Customer_Txn_Count'] = df['customer_id'].map(customer_count)
df[['membership_type', 'Avg_Purchase_By_Membership', 'customer_id', 'Customer_Total_Purchase', 'Customer_Txn_Count']].head()

,membership_type,Avg_Purchase_By_Membership,customer_id,Customer_Total_Purchase,Customer_Txn_Count
0,Gold,124.792169,100508,99.13,1
1,Silver,128.148626,100819,113.23,1
2,Gold,124.792169,100453,248.19,1
3,Silver,128.148626,100369,51.27,1
4,Silver,128.148626,100243,97.97,1


**Code Explanation:** A small summary table is built for each group using groupby, then map applies that summary back onto every row. Avg_Purchase_By_Membership shows the average spend within each membership tier, while Customer_Total_Purchase and Customer_Txn_Count summarize each customer's total spend and number of transactions.

## 8. Create Interaction Features

**Definition:** Combining two existing features together to capture a relationship that neither one shows alone.

**Example:** Income_x_Quantity, multiplying income by quantity purchased, to see if higher earners also tend to buy more items at once.

**Why it is used?** Some patterns only appear when two features are considered together, not separately.

In [9]:
df['Income_x_Quantity'] = df['annual_income'] * df['quantity']
df['Age_x_Rating'] = df['age'] * df['rating']
df[['annual_income', 'quantity', 'Income_x_Quantity', 'age', 'rating', 'Age_x_Rating']].head()

,annual_income,quantity,Income_x_Quantity,age,rating,Age_x_Rating
0,80242.98,1.0,80242.98,43.0,1.0,43.0
1,65884.34,1.0,65884.34,41.0,4.0,164.0
2,56285.40,2.0,112570.80,61.0,4.0,244.0
3,81878.74,9.0,736908.66,38.0,4.0,152.0
4,100566.42,1.0,100566.42,24.0,4.0,96.0


**Code Explanation:** Income_x_Quantity multiplies annual_income by quantity, and Age_x_Rating multiplies age by rating. Age_x_Rating will turn out to matter later, since it is built directly from the same rating column used to define the target.

## 9. Perform Feature Selection

**Definition:** Scoring every candidate feature to see how strongly it relates to the target, before deciding what to keep.

**Example:** A target called High_Rating is created, marking 1 if rating is 4 or 5 and 0 otherwise, and every feature is scored against it.

**Why it is used?** Narrows a large feature set down to the ones that actually carry a signal worth keeping.

In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
df['Price_Per_Item'] = df['purchase_amount'] / df['quantity']
df['Income_To_Purchase_Ratio'] = df['annual_income'] / df['purchase_amount']
snapshot_date = df['signup_date'].max() + pd.Timedelta(days=1)
df['Tenure_Days'] = (snapshot_date - df['signup_date']).dt.days
membership_avg = df.groupby('membership_type')['purchase_amount'].mean()
df['Avg_Purchase_By_Membership'] = df['membership_type'].map(membership_avg)
customer_total = df.groupby('customer_id')['purchase_amount'].sum()
df['Customer_Total_Purchase'] = df['customer_id'].map(customer_total)
customer_count = df.groupby('customer_id')['customer_id'].count()
df['Customer_Txn_Count'] = df['customer_id'].map(customer_count)
df['Income_x_Quantity'] = df['annual_income'] * df['quantity']
df['Age_x_Rating'] = df['age'] * df['rating']
df['High_Rating'] = (df['rating'] >= 4).astype(int)
text_columns = {
    'gender': 'gender_code',
    'city': 'city_code',
    'membership_type': 'membership_code',
    'payment_method': 'payment_code'
}
encoder = LabelEncoder()
for source_col, new_col in text_columns.items():
    df[new_col] = encoder.fit_transform(df[source_col])
candidate_features = [
    'age', 'annual_income', 'purchase_amount', 'quantity',
    'gender_code', 'city_code', 'membership_code', 'payment_code',
    'Price_Per_Item', 'Income_To_Purchase_Ratio', 'Tenure_Days',
    'Avg_Purchase_By_Membership', 'Customer_Total_Purchase', 'Customer_Txn_Count',
    'Income_x_Quantity', 'Age_x_Rating'
]
X = df[candidate_features]
y = df['High_Rating']
selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X, y)
scores = pd.Series(selector.scores_, index=candidate_features)
scores = scores.sort_values(ascending=False)
scores

Age_x_Rating                  1104.044097
Customer_Txn_Count               5.476829
Customer_Total_Purchase          3.045185
annual_income                    1.407716
age                              1.356770
Income_x_Quantity                1.022585
purchase_amount                  0.759860
quantity                         0.703342
gender_code                      0.652335
Tenure_Days                      0.565865
payment_code                     0.548054
Avg_Purchase_By_Membership       0.412716
city_code                        0.398793
Price_Per_Item                   0.355317
Income_To_Purchase_Ratio         0.125537
membership_code                  0.080506
dtype: float64

**Code Explanation:** A loop label-encodes every text column into numbers instead of repeating the same line four times. A High_Rating target is created from rating, and SelectKBest scores every candidate feature. Age_x_Rating scores wildly higher than everything else, which is the first warning sign of leakage.

## 10. Analyze Feature Importance

**Definition:** Training a real model and checking which features it actually relied on to make predictions.

**Example:** A Random Forest trained on all 16 candidate features to predict High_Rating.

**Why it is used?** Confirms or challenges what the filter-based selection scores suggested, using an actual trained model.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
importances = pd.Series(model.feature_importances_, index=candidate_features)
importances = importances.sort_values(ascending=False)
importances.head(6)

Age_x_Rating                0.563252
age                         0.127237
Income_To_Purchase_Ratio    0.036904
annual_income               0.035265
purchase_amount             0.034216
Customer_Total_Purchase     0.033032
dtype: float64

In [12]:
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)

0.975

**Code Explanation:** Age_x_Rating alone accounts for over half of the model's total feature importance, and the model reaches a suspiciously high accuracy, both strong signs that something in the feature set is leaking the target.

## 11. Check for Feature Leakage

**Definition:** Investigating suspiciously strong features to confirm whether they are genuinely predictive or secretly built from the target itself.

**Example:** Age_x_Rating is directly calculated using the rating column, and High_Rating is also directly calculated from rating, so Age_x_Rating is mathematically tied to the target it is supposed to help predict.

**Why it is used?** Confirms whether the high accuracy above is real skill or just the model reading the answer through a disguised feature.

In [14]:
safe_features = candidate_features.copy()
safe_features.remove('Age_x_Rating')
X_safe = df[safe_features]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_safe, y, test_size=0.2, random_state=42)
model2 = RandomForestClassifier(n_estimators=200, random_state=42)
model2.fit(X_train2, y_train2)
predictions2 = model2.predict(X_test2)
accuracy_score(y_test2, predictions2)

0.575

**Code Explanation:** Removing only Age_x_Rating drops accuracy sharply, confirming it was leaking the target. The lower score afterward is a much more honest, believable number for how well these features can really predict High_Rating.

## 12. Remove Unnecessary Features

**Definition:** Dropping features that are leaky, redundant, or too weak to be worth keeping in the final dataset.

**Example:** Dropping Age_x_Rating for leakage, and dropping weak filter-score features like membership_code and Income_To_Purchase_Ratio for being close to irrelevant.

**Why it is used?** A smaller, cleaner feature set is easier to maintain, faster to train, and free of hidden shortcuts a model could exploit unfairly.

In [15]:
features_to_remove = [
    'Age_x_Rating', 'membership_code', 'Income_To_Purchase_Ratio', 'city_code',
    'gender_code', 'payment_code', 'Avg_Purchase_By_Membership', 'Income_x_Quantity'
]
kept_features = []
for feature in candidate_features:
    if feature not in features_to_remove:
        kept_features.append(feature)
kept_features

['age',
 'annual_income',
 'purchase_amount',
 'quantity',
 'Price_Per_Item',
 'Tenure_Days',
 'Customer_Total_Purchase',
 'Customer_Txn_Count']

**Code Explanation:** A simple loop keeps only the features that are not in the removal list. Age_x_Rating is removed for leakage, and several low-scoring categorical and ratio features are removed for weak relevance.

## 13. Create Final Feature Dataset

**Definition:** Assembling the kept features, plus the target, into one final table ready to be used for modeling.

**Example:** A dataset with age, annual_income, purchase_amount, quantity, Price_Per_Item, Tenure_Days, Customer_Total_Purchase, Customer_Txn_Count, and High_Rating.

**Why it is used?** This is the actual deliverable of the whole workflow, the dataset a model would be trained on going forward.

In [15]:
final_columns = kept_features + ['High_Rating']
final_df = df[final_columns]
final_df.head()

,age,annual_income,purchase_amount,quantity,Price_Per_Item,Tenure_Days,Customer_Total_Purchase,Customer_Txn_Count,High_Rating
0,43.0,80242.98,99.13,1.0,99.130000,861,99.13,1,0
1,41.0,65884.34,113.23,1.0,113.230000,1293,113.23,1,1
2,61.0,56285.40,248.19,2.0,124.095000,1189,248.19,1,1
3,38.0,81878.74,51.27,9.0,5.696667,112,51.27,1,1
4,24.0,100566.42,97.97,1.0,97.970000,913,97.97,1,1


In [16]:
final_df.shape

(1000, 9)

**Code Explanation:** final_df pulls together the kept features plus the High_Rating target into one clean table, shrinking the dataset from its original 12 raw columns down to a small, focused, ML-ready set.

## 14. Document All Feature Engineering Decisions

| Feature | Stage Created | Decision | Reason |
|---|---|---|---|
| gender, city cleanup | Clean Dataset | Kept | Fixed inconsistent casing and spelling |
| purchase_amount, age fixes | Clean Dataset | Kept | Removed currency symbols and impossible values |
| Price_Per_Item | Numerical | Kept | Meaningful per-item cost signal |
| Income_To_Purchase_Ratio | Numerical | Removed | Very low feature selection score |
| Income_Bracket, Age_Group | Categorical | Kept for reporting | Useful for readability, not used in the final model to avoid duplicating age and annual_income |
| Signup_Year, Signup_Month | Date/Time | Removed | Tenure_Days already captures the useful time signal more directly |
| Tenure_Days | Date/Time | Kept | Safe, meaningful customer lifetime signal |
| Avg_Purchase_By_Membership | Aggregation | Removed | Weak individual score, largely duplicated by membership_code |
| Customer_Total_Purchase | Aggregation | Kept | Reasonable predictive signal, no leakage since it is not target-derived |
| Customer_Txn_Count | Aggregation | Kept | Highest non-leaky selection score among engineered features |
| Income_x_Quantity | Interaction | Removed | Weak score, largely redundant with annual_income and quantity separately |
| Age_x_Rating | Interaction | Removed | Confirmed leakage, built directly from the target-related rating column |
